# Пример - KVP-разложение

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **efb5bdb**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/efb5bdb8bb68b7fe975141ef21b1a9cbcdfec5f2.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Девять расчётов проверяют разложения таблиц пар, дистанционный переход, метрические параметры и знаковые лесные топономы.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra efb5bdb](https://github.com/magin67/polyform_algebra/commit/efb5bdb8bb68b7fe975141ef21b1a9cbcdfec5f2).


## Разложение пути

На четырёхвершинном пути строим K, V и P. Проверяем реконструкцию, нулевые суммы строк P и ортогональность компонент по коэффициентам.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D0%B0%D1%80%D0%BD%D1%8B%D1%85-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA#%D0%BA%D0%B0%D0%BA-%D1%8D%D1%82%D0%BE-%D0%B2%D1%8B%D0%B3%D0%BB%D1%8F%D0%B4%D0%B8%D1%82-%D0%BD%D0%B0-%D0%BF%D1%83%D1%82%D0%B8-%D0%B8%D0%B7-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD).

In [3]:
from itertools import combinations

from sympy import Matrix, Rational, expand, eye, ones, symbols, zeros

from polyform_algebra.core import (
    circumsphere, form_exp, green, laplacian_form, network,
    partition_form, potential, resistance, spanning_tree_weight,
)
from polyform_algebra.kvp import decompose, distance_table, pair_laplacian, signed_forest_toponome
from polyform_algebra.toponomes import vertex_toponome

labels = tuple("abcd")
pairs = tuple(combinations(range(4), 2))


def pair_values(table):
    return tuple(table[i, j] for i, j in pairs)


def graph(edges):
    return network(labels, tuple((a, b, weight) for (a, b), weight in edges))


def couplings(net):
    table = zeros(len(net.context.labels))
    for a, b, weight in net.edges:
        i, j = map(net.context.labels.index, (a, b))
        table[i, j] = table[j, i] = weight
    return table


path = graph((("ab", 1), ("bc", 1), ("cd", 1)))
C = couplings(path)
parts = decompose(C)
assert parts.K + parts.V + parts.P == C
assert parts.P * ones(4, 1) == zeros(4, 1)
assert pair_values(parts.P) == tuple(Rational(x, 2) for x in (1, -1, 0, 0, -1, 1))
assert all(sum(a.multiply_elementwise(b)) == 0 for a, b in combinations((parts.K, parts.V, parts.P), 2))
print("P₄ pair residual:", pair_values(parts.P))

P₄ pair residual: (1/2, -1/2, 0, 0, -1/2, 1/2)


## Звезда, цикл и взвешенный граф

На звезде и цикле проверяем исчезновение P и V соответственно. На взвешенном графе сверяем P по всем парам и сокращение трёх вкладов на отсутствующем ребре.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/kvp-%D0%BD%D0%B0-%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%8B%D1%85-%D0%B8-%D0%B2%D0%B7%D0%B2%D0%B5%D1%88%D0%B5%D0%BD%D0%BD%D1%8B%D1%85-%D0%B3%D1%80%D0%B0%D1%84%D0%B0%D1%85#%D0%B3%D1%80%D0%B0%D1%84-%D1%81-%D0%BD%D0%B5%D1%80%D0%B0%D0%B2%D0%BD%D1%8B%D0%BC%D0%B8-%D0%B2%D0%B5%D1%81%D0%B0%D0%BC%D0%B8).

In [4]:
star = graph((("ab", 1), ("ac", 1), ("ad", 1)))
cycle = graph((("ab", 1), ("bc", 1), ("cd", 1), ("ad", 1)))
weighted = graph((("ab", 4), ("ac", 2), ("bc", 1), ("bd", 3), ("cd", 2)))
assert decompose(couplings(star)).P == zeros(4)
assert decompose(couplings(cycle)).V == zeros(4)
weighted_parts = decompose(couplings(weighted))
assert pair_values(weighted_parts.P) == tuple(Rational(x, 2) for x in (2, 1, -3, -3, 1, 2))
assert weighted_parts.K[0, 3] + weighted_parts.V[0, 3] + weighted_parts.P[0, 3] == 0
print("Weighted P component:", pair_values(weighted_parts.P))

Weighted P component: (1, 1/2, -3/2, -3/2, 1/2, 1)


## Одна дуга и пять компонент

Для одной дуги с изолированной вершиной проверяем все пять компонент. Остаток O образует треугольную циркуляцию в таблице пар.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/%D0%BE%D1%80%D0%B8%D0%B5%D0%BD%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5-%D1%80%D0%B0%D1%81%D1%88%D0%B8%D1%80%D0%B5%D0%BD%D0%B8%D1%8F-kvp---%D1%86%D0%B8%D1%80%D0%BA%D1%83%D0%BB%D1%8F%D1%86%D0%B8%D1%8F-%D0%B8-%D0%B4%D0%B8%D1%81%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%BE%D1%80%D0%B8%D0%B5%D0%BD%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D1%8B%D1%85-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2).

In [5]:
arc = Matrix([[0, 1, 0], [0, 0, 0], [0, 0, 0]])
oriented = decompose(arc)
assert oriented.K + oriented.V + oriented.P + oriented.O + oriented.U == arc
assert oriented.P == zeros(3)
assert oriented.O == Matrix([[0, 1, -1], [-1, 0, 1], [1, -1, 0]]) / 6
assert oriented.U[0, 1] == Rational(1, 3)
assert oriented.O * ones(3, 1) == zeros(3, 1)
print("One arc: O_ab =", oriented.O[0, 1], "| U_ab =", oriented.U[0, 1])

One arc: O_ab = 1/6 | U_ab = 1/3


## Дистанционный оператор

Применяем дистанционный оператор к лапласианам пути, направленного цикла и одной дуги. Проверяем множители компонент при балансе и вклад U в V без баланса.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/%D0%B4%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9-%D0%BE%D0%BF%D0%B5%D1%80%D0%B0%D1%82%D0%BE%D1%80-%D0%B8-kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BF%D1%83%D1%82%D1%8C-%D0%BD%D0%B0-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [6]:
Q = distance_table(pair_laplacian(C))
assert Q == 8 * parts.K + 4 * parts.V + 2 * parts.P
assert pair_values(Q) == (5, 3, 2, 6, 3, 5)
cycle_arcs = Matrix([[0, 1, 0], [0, 0, 1], [1, 0, 0]])
cycle_parts = decompose(cycle_arcs)
assert cycle_parts.U == zeros(3)
assert distance_table(pair_laplacian(cycle_arcs)) == 6 * cycle_parts.K + 2 * cycle_parts.O
one_arc_image = decompose(distance_table(pair_laplacian(arc)))
assert one_arc_image.V == 3 * oriented.V + Matrix([[0, 0, 1], [0, 0, -1], [1, -1, 0]]) / 2
print("Distance image of P₄:", pair_values(Q))

Distance image of P₄: (5, 3, 2, 6, 3, 5)


## От гриниана к резистенсам

Строим парную таблицу по внедиагональным элементам гриниана. Сравниваем её компоненты с разложением резистенсов и проверяем индекс Кирхгофа.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%86%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2#%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D1%8B-%D1%81%D1%80%D0%B5%D0%B4%D0%BD%D0%B5%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%BD%D1%8B%D0%B9-%D0%B2%D0%BA%D0%BB%D0%B0%D0%B4-%D0%B8-%D0%BE%D1%81%D1%82%D0%B0%D1%82%D0%BE%D0%BA).

In [7]:
Gr = Matrix(green(path).value.values)
gamma = Matrix(4, 4, lambda i, j: -Gr[i, j] if i != j else 0)
R = Matrix(resistance(path).value.values)
g_parts, r_parts = decompose(gamma), decompose(R)
assert pair_laplacian(gamma) == Gr and distance_table(Gr) == R
assert (r_parts.K, r_parts.V, r_parts.P) == (8 * g_parts.K, 4 * g_parts.V, 2 * g_parts.P)
assert sum(pair_values(R)) == 4 * Gr.trace() == 10
print("Green table to resistance: multipliers 8, 4, 2; Kf = 10")

Green table to resistance: multipliers 8, 4, 2; Kf = 10


## Контакты и изменение проводимости

Добавляем к циклу четыре контактных сопротивления и считаем резистенсы между внешними клеммами. Затем отдельно меняем внутреннюю проводимость и проверяем изменение P.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F---%D1%8D%D0%BB%D0%B5%D0%BA%D1%82%D1%80%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7-%D0%BF%D0%B0%D1%80%D0%BD%D1%8B%D1%85-%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85#%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B9-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BA%D0%BE%D0%BD%D1%82%D0%B0%D0%BA%D1%82%D1%8B-%D0%B8-%D0%B8%D0%B7%D0%BC%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D1%80%D0%BE%D0%B2%D0%BE%D0%B4%D0%B8%D0%BC%D0%BE%D1%81%D1%82%D0%B8).

In [8]:
base = Matrix(resistance(cycle).value.values)
terminals = tuple("xyzw")
contacts = tuple(Rational(i, 4) for i in range(1, 5))
extended = network(labels + terminals, cycle.edges + tuple(
    (a, terminal, 1 / contact) for a, terminal, contact in zip(labels, terminals, contacts)
))
terminal_pairs = tuple((terminals[i], terminals[j]) for i, j in pairs)
values = resistance(extended, pairs=terminal_pairs).value.values
terminal_table = zeros(4)
for (i, j), value in zip(pairs, values):
    terminal_table[i, j] = terminal_table[j, i] = value
assert all(terminal_table[i, j] == base[i, j] + contacts[i] + contacts[j] for i, j in pairs)
assert decompose(terminal_table).P == decompose(base).P
changed = graph((("ab", 2), ("bc", 1), ("cd", 1), ("ad", 1)))
delta_P = decompose(Matrix(resistance(changed).value.values) - base).P
assert pair_values(delta_P) == tuple(Rational(x, 84) for x in (-5, -2, 7, 7, -2, -5))
print("Contacts preserve P. Internal change:", pair_values(delta_P))

Contacts preserve P. Internal change: (-5/84, -1/42, 1/12, 1/12, -1/42, -5/84)


## Одинаковые K и V и разные радиусы

На двух графах из заметки сравниваем K, V и P резистенсов, вычисляем радиусы и остовные коэффициенты.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/kvp-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B8-%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BA%D0%B0%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BE%D0%B4%D0%B8%D0%BD%D0%B0%D0%BA%D0%BE%D0%B2%D1%8B%D0%B5-k-%D0%B8-v-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2-%D1%80%D0%B0%D0%B7%D0%BD%D1%8B%D0%B5-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81%D1%8B).

In [9]:
weights = (
    (Rational(1, 4), Rational(1, 4), Rational(3, 20), Rational(5, 12), Rational(1, 4), Rational(1, 4)),
    tuple(Rational(x, 51) for x in (26, 8, 4, 16, 8, 26)),
)
graphs = [graph((labels[i] + labels[j], w) for (i, j), w in zip(pairs, row)) for row in weights]
first, second = [decompose(Matrix(resistance(net).value.values)) for net in graphs]
assert first.K == second.K and first.V == second.V and first.P != second.P
spheres = [circumsphere(net).value for net in graphs]
radii = tuple(sphere.resistance_radius for sphere in spheres)
assert radii == (Rational(49, 64), Rational(37, 48))
assert tuple(spanning_tree_weight(net).value for net in graphs) == (Rational(4, 15), Rational(16, 51))
print("Same resistance K and V, radii:", ", ".join(map(str, radii)))

Same resistance K and V, radii: 49/64, 37/48


## Симметрический потенциал второго порядка

Для пути с проводимостями 1, 2, 3 вычисляем потенциалы четырёх троек. Сравниваем сумму после нормировки с квадратичной формулой через K, V и P.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B3%D0%BE-%D0%BF%D0%BE%D1%80%D1%8F%D0%B4%D0%BA%D0%B0-%D0%B8-kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%BD%D0%B0-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [10]:
weighted_path = graph((("ab", 1), ("bc", 2), ("cd", 3)))
metric = form_exp(laplacian_form(weighted_path).value).value
triples = combinations(labels, 3)
raw_potentials = [
    potential(metric, partition_form(weighted_path.context, (triple,)).value).value
    for triple in triples
]
tau = spanning_tree_weight(weighted_path).value
assert tau == 6
assert tuple(value / tau for value in raw_potentials) == tuple(
    Rational(x, 6) for x in (3, 5, 3, 1)
)
resistance_parts = decompose(Matrix(resistance(weighted_path).value.values))
norms = [sum(component[i, j] ** 2 for i, j in pairs)
         for component in (resistance_parts.K, resistance_parts.V, resistance_parts.P)]
assert norms[0] / 2 - norms[1] / 2 - norms[2] == sum(raw_potentials) / tau == 2
assert sum(raw_potentials) == 12
print("Weighted path: nu₂ = 2, U₂ = 12")

Weighted path: nu₂ = 2, U₂ = 12


## Лесной топоном P-компоненты

Перечисляем леса со знаковыми весами для P-компоненты пятивершинного пути. Проверяем сокращение типа (2,1), корневую нормировку и равенство с определителем.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/kvp-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5/%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%82%D0%BE%D0%BF%D0%BE%D0%BD%D0%BE%D0%BC%D1%8B-kvp-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---p-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D0%B0-%D0%BF%D1%83%D1%82%D0%B8-%D0%BD%D0%B0-%D0%BF%D1%8F%D1%82%D0%B8-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [11]:
C5 = zeros(5)
for i in range(4):
    C5[i, i + 1] = C5[i + 1, i] = 1
components = decompose(C5)
forest_types = signed_forest_toponome(components.P)
assert forest_types == {
    (): 1, (2,): -2, (1, 1): 1, (3,): Rational(-2, 27), (4,): Rational(1, 9),
}
t = symbols("t")
rooted = sum(value * t ** sum(kind) for kind, value in vertex_toponome(forest_types).items())
assert rooted == 1 - 2 * t**2 - Rational(8, 27) * t**3 + Rational(5, 9) * t**4
assert rooted == expand((eye(5) + t * pair_laplacian(components.P)).det())
full_types = signed_forest_toponome(C5)
for kind in ((2,), (1, 1)):
    assert sum(signed_forest_toponome(part).get(kind, 0)
               for part in (components.K, components.V, components.P)) == full_types[kind] == 3
print("Signed P₅ forest type (2,1) cancels; rooted polynomial agrees with determinant")

Signed P₅ forest type (2,1) cancels; rooted polynomial agrees with determinant